<a href="https://colab.research.google.com/github/DeepFluxion/2026_2_IBMEC_PROG_ANALISE_DADOS/blob/main/notebooks/Aula03_Revisado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://keystoneacademic-res.cloudinary.com/image/upload/c_pad,w_640,h_304/dpr_auto/f_auto/q_auto/v1/element/94/94774_thumb.png" width=300>

# Programação para Análise de Dados

## Aula 3: Fundamentos do pandas e leitura de dados.

### Professor: Carlos E. Leal de Castro e Domingos Napolitano

### ANTES!

- Sentem nos mesmos computadores que vocês configuraram na aula passada! Liguem os Computadores! Façam o login!

- Se for o seu computador ou o mesmo, o processo de criação do nosso Env já deveria estar instalado!

- Abram (ou baixem) o Visual Code Studio! Vocês vão acompanhar a aula por lá!

- Caso queiram, abram o Prompt do Anaconda e digitem:

```bash
conda activate kernelPAD && jupyter notebook
```

### Objetivos de Aprendizagem
- Compreender as estruturas **Series** e **DataFrame** do `pandas`.
- Ler dados de arquivos **CSV** e **Excel**.
- Realizar **exploração inicial** dos dados (`head`, `tail`, `info`, `describe`).
- Executar **seleção e filtragem** básica (`[]`, `.loc`, `.iloc`, máscaras booleanas e `query`).

> Slides marcados como **EXERCÍCIO** devem ser feitos em sala

## Aquecimento: o que é transformação de dados?

Transformação de dados é o processo de **organizar, limpar e padronizar** dados para análise.  
Nesta aula, focaremos no **primeiro contato**: carregar, inspecionar e fazer seleções simples.

## Setup do Ambiente

In [1]:
import pandas as pd
import numpy as np

# Caminho para os dados usados nos exercícios (ajuste se necessário)
DATA_DIR = "."
#CLIENTES_CSV = "dados_clientes_ptbr.csv"
CLIENTES_CSV = 'https://raw.githubusercontent.com/DeepFluxion/2026_2_IBMEC_PROG_ANALISE_DADOS/refs/heads/main/datasets/dados_clientes_ptbr.csv'
#VENDAS_CSV   = "dados_vendas.csv"
VENDAS_CSV   = "https://raw.githubusercontent.com/DeepFluxion/2026_2_IBMEC_PROG_ANALISE_DADOS/refs/heads/main/datasets/dados_vendas.csv"

## Estruturas de dados do pandas

**Series** é um vetor rotulado unidimensional.  
**DataFrame** é uma tabela bidimensional com rótulos (linhas/colunas).

Características importantes:
- Rótulos (`index` e `columns`) para alinhamento automático.
- Operações **vetorizadas** (rápidas) e funções convenientes.
- Suporte a tipos heterogêneos por coluna.

### Criando uma `Series`

In [2]:
s = pd.Series([10, 20, 30], index=["a", "b", "c"], name="minha_serie")
print(s)

a    10
b    20
c    30
Name: minha_serie, dtype: int64


### Criando um `DataFrame`

In [3]:
dados = {
    "produto": ["A", "B", "C", "D", "E"],
    "preco": [10.0, 15.5, 8.7, 42.0, 66.6],
    "estoque": [100, 50, 0, 5, 13],
}
df = pd.DataFrame(dados)
df

,produto,preco,estoque
0,A,10.0,100
1,B,15.5,50
2,C,8.7,0
3,D,42.0,5
4,E,66.6,13


### Atributos úteis

In [4]:
df.shape, df.dtypes, df.index, df.columns

((5, 3),
 produto     object
 preco      float64
 estoque      int64
 dtype: object,
 RangeIndex(start=0, stop=5, step=1),
 Index(['produto', 'preco', 'estoque'], dtype='object'))

## Leitura de Arquivos CSV

#### Dica rápida
- CSV em pt-BR muitas vezes usa **;** como separador e **,** como separador decimal.  
- Use `sep=';'` e `decimal=','`.  
- Caso seu arquivo esteja em `latin-1`/`ISO-8859-1`, use `encoding='latin-1'`.

In [5]:
# Lendo um CSV pt-BR (fornecido com este notebook)
df_clientes = pd.read_csv(CLIENTES_CSV, sep=';', decimal=',', encoding='utf-8')
df_clientes.tail(2)

,id,nome,cidade,data_cadastro,idade,renda_mensal,limite_credito
3,4,Diego,Belo Horizonte,30/09/2022,23,"3.200,00","8.000,00"
4,5,Eva,Curitiba,11/02/2025,30,"9.800,95","25.000,00"


In [6]:
# Parâmetros essenciais do read_csv (demonstração)
amostra = pd.read_csv(
    CLIENTES_CSV,
    sep=';',
    decimal=',',
    usecols=['id', 'nome', 'cidade', 'idade', 'renda_mensal'],
    dtype={'id': 'int64', 'idade': 'Int64'},
    nrows=3,
#     parse_dates=['data_cadastro'],  # Ignorado aqui porque o CSV tem data dd/mm/yyyy como texto
    dayfirst=True,                  # útil para dd/mm/yyyy quando parse_dates estiver ativo
    engine='python'                 # útil em arquivos complexos
)
amostra

,id,nome,cidade,idade,renda_mensal
0,1,Ana,São Paulo,28,"4.500,50"
1,2,Bruno,Rio de Janeiro,35,"8.200,00"
2,3,Carla,Recife,41,"5.100,75"


In [7]:
# Ajuste de datas quando o parse_dates não funcionar direto (ex: dd/mm/yyyy em coluna texto)
df_clientes['data_cadastro'] = pd.to_datetime(df_clientes['data_cadastro'],
                                              format='%d/%m/%Y')

df_clientes['coluna_teste'] = [1,2,3,4,5]
# df_clientes.dtypes
df_clientes.head()

,id,nome,cidade,data_cadastro,idade,renda_mensal,limite_credito,coluna_teste
0,1,Ana,São Paulo,2024-01-15,28,"4.500,50","12.000,00",1
1,2,Bruno,Rio de Janeiro,2023-11-03,35,"8.200,00","20.000,00",2
2,3,Carla,Recife,2024-05-21,41,"5.100,75","15.000,00",3
3,4,Diego,Belo Horizonte,2022-09-30,23,"3.200,00","8.000,00",4
4,5,Eva,Curitiba,2025-02-11,30,"9.800,95","25.000,00",5


## Leitura de Arquivos Excel (XLS/XLSX)

Para ler `.xlsx`, instale o motor adequado (ex.: `openpyxl`).  
Ex.: `pip install openpyxl`

Parâmetros úteis: `sheet_name`, `usecols`, `skiprows`, `nrows`, `dtype`, `parse_dates`.

In [8]:
!pip install openpyxl
# EXECUTE

In [29]:
df_clientes.to_excel('df_cliente.xlsx')

In [31]:
# Exemplo (ajuste o caminho para um arquivo .xlsx que você possua):
df_excel = pd.read_excel("/content/df_cliente.xlsx")
df_excel.head()
#print("Exemplo comentado: use pd.read_excel('arquivo.xlsx') quando tiver um arquivo Excel disponível.")

,Unnamed: 0,id,nome,cidade,data_cadastro,idade,renda_mensal,limite_credito,coluna_teste
0,0,1,Ana,São Paulo,2024-01-15,28,"4.500,50","12.000,00",1
1,1,2,Bruno,Rio de Janeiro,2023-11-03,35,"8.200,00","20.000,00",2
2,2,3,Carla,Recife,2024-05-21,41,"5.100,75","15.000,00",3
3,3,4,Diego,Belo Horizonte,2022-09-30,23,"3.200,00","8.000,00",4
4,4,5,Eva,Curitiba,2025-02-11,30,"9.800,95","25.000,00",5


## Exploração Inicial dos Dados

Para ler seus dados no Google Drive, faça o seguinte:`
- Seu link de compartilhamento, provavelmente, é assim: `https://drive.google.com/file/d/1kQmDMkMvQwYjvcG9Y52AfPm-6KTIZWKP/view?usp=drive_link`
- O ID do arquivo é: `1kQmDMkMvQwYjvcG9Y52AfPm-6KTIZWKP`

Com isso, basta montar o link assim:

- `https://drive.google.com/uc?export=download&id=ID_DO_ARQUIVO`

In [32]:
# ID do arquivo no Google Drive
file_id = "1kQmDMkMvQwYjvcG9Y52AfPm-6KTIZWKP"
# Link direto de download
url = f"https://drive.google.com/uc?export=download&id={file_id}"
# Lendo o CSV
df = pd.read_csv(url)
# Mostrando as 5 primeiras linhas
df.head()

,genre,artist_name,track_name,track_id,popularity,acousticness,danceability,duration_ms,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,time_signature,valence
0,Movie,Henri Salvador,C'est beau de faire un Show,0BRjO6ga9RKCKjfDqeFgWV,0,0.611,0.389,99373,0.910,0.000,C#,0.3460,-1.828,Major,0.0525,166.969,4/4,0.814
1,Movie,Martin & les fées,Perdu d'avance (par Gad Elmaleh),0BjC1NfoEOOusryehmNudP,1,0.246,0.590,137373,0.737,0.000,F#,0.1510,-5.559,Minor,0.0868,174.003,4/4,0.816
2,Movie,Joseph Williams,Don't Let Me Be Lonely Tonight,0CoSDzoNIKCRs124s9uTVy,3,0.952,0.663,170267,0.131,0.000,C,0.1030,-13.879,Minor,0.0362,99.488,5/4,0.368
3,Movie,Henri Salvador,Dis-moi Monsieur Gordon Cooper,0Gc6TVm52BwZD07Ki6tIvf,0,0.703,0.240,152427,0.326,0.000,C#,0.0985,-12.178,Major,0.0395,171.758,4/4,0.227
4,Movie,Fabien Nataf,Ouverture,0IuslXpMROHdEPvSl1fTQK,4,0.950,0.331,82625,0.225,0.123,F,0.2020,-21.150,Major,0.0456,140.576,4/4,0.390


In [33]:
df.tail(3) # Mostra as ultimas 3 linhas

,genre,artist_name,track_name,track_id,popularity,acousticness,danceability,duration_ms,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,time_signature,valence
232722,Soul,Muddy Waters,(I'm Your) Hoochie Coochie Man,2ziWXUmQLrXTiYjCg2fZ2t,47,0.9010,0.517,166960,0.419,0.000000,D,0.0945,-8.282,Major,0.1480,84.135,4/4,0.813
232723,Soul,R.LUM.R,With My Words,6EFsue2YbIG4Qkq8Zr9Rir,44,0.2620,0.745,222442,0.704,0.000000,A,0.3330,-7.137,Major,0.1460,100.031,4/4,0.489
232724,Soul,Mint Condition,You Don't Have To Hurt No More,34XO9RwPMKjbvRry54QzWn,35,0.0973,0.758,323027,0.470,0.000049,G#,0.0836,-6.708,Minor,0.0287,113.897,4/4,0.479


In [34]:
df.info()  # tipos, nulos e memória

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 232725 entries, 0 to 232724
Data columns (total 18 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   genre             232725 non-null  object 
 1   artist_name       232725 non-null  object 
 2   track_name        232724 non-null  object 
 3   track_id          232725 non-null  object 
 4   popularity        232725 non-null  int64  
 5   acousticness      232725 non-null  float64
 6   danceability      232725 non-null  float64
 7   duration_ms       232725 non-null  int64  
 8   energy            232725 non-null  float64
 9   instrumentalness  232725 non-null  float64
 10  key               232725 non-null  object 
 11  liveness          232725 non-null  float64
 12  loudness          232725 non-null  float64
 13  mode              232725 non-null  object 
 14  speechiness       232725 non-null  float64
 15  tempo             232725 non-null  float64
 16  time_signature    23

In [35]:
df.describe()  # estatísticas de colunas numéricas

,popularity,acousticness,danceability,duration_ms,energy,instrumentalness,liveness,loudness,speechiness,tempo,valence
count,232725.000000,232725.000000,232725.000000,2.327250e+05,232725.000000,232725.000000,232725.000000,232725.000000,232725.000000,232725.000000,232725.000000
mean,41.127502,0.368560,0.554364,2.351223e+05,0.570958,0.148301,0.215009,-9.569885,0.120765,117.666585,0.454917
std,18.189948,0.354768,0.185608,1.189359e+05,0.263456,0.302768,0.198273,5.998204,0.185518,30.898907,0.260065
min,0.000000,0.000000,0.056900,1.538700e+04,0.000020,0.000000,0.009670,-52.457000,0.022200,30.379000,0.000000
25%,29.000000,0.037600,0.435000,1.828570e+05,0.385000,0.000000,0.097400,-11.771000,0.036700,92.959000,0.237000
50%,43.000000,0.232000,0.571000,2.204270e+05,0.605000,0.000044,0.128000,-7.762000,0.050100,115.778000,0.444000
75%,55.000000,0.722000,0.692000,2.657680e+05,0.787000,0.035800,0.264000,-5.501000,0.105000,139.054000,0.660000
max,100.000000,0.996000,0.989000,5.552917e+06,0.999000,0.999000,1.000000,3.744000,0.967000,242.903000,1.000000


In [36]:
# Outras explorações úteis
df['artist_name'].value_counts()

,count
artist_name,
Giuseppe Verdi,1394
Giacomo Puccini,1137
Kimbo Children's Music,971
Nobuo Uematsu,825
Richard Wagner,804
...,...
Major Harris,1
The Magnolia,1
Scienze,1


## Seleção e Filtragem Básica (25 min)

### Selecionando colunas e linhas
- `df['col']` ou `df[['col1','col2']]`
- `.loc[linhas, colunas]` por **rótulo**
- `.iloc[linhas, colunas]` por **posição**

In [37]:
# Colunas
df['artist_name'].head()

,artist_name
0,Henri Salvador
1,Martin & les fées
2,Joseph Williams
3,Henri Salvador
4,Fabien Nataf


In [38]:
df[['artist_name', 'popularity']].head()

,artist_name,popularity
0,Henri Salvador,0
1,Martin & les fées,1
2,Joseph Williams,3
3,Henri Salvador,0
4,Fabien Nataf,4


In [39]:
# Linhas por rótulo vs posição
df.loc[0:8, ['artist_name', 'popularity', 'danceability']]

,artist_name,popularity,danceability
0,Henri Salvador,0,0.389
1,Martin & les fées,1,0.590
2,Joseph Williams,3,0.663
3,Henri Salvador,0,0.240
4,Fabien Nataf,4,0.331
5,Henri Salvador,0,0.578
6,Martin & les fées,2,0.703
7,Laura Mayne,15,0.416
8,Chorus,0,0.734


In [40]:
df.iloc[0:3, 0:5]

,genre,artist_name,track_name,track_id,popularity
0,Movie,Henri Salvador,C'est beau de faire un Show,0BRjO6ga9RKCKjfDqeFgWV,0
1,Movie,Martin & les fées,Perdu d'avance (par Gad Elmaleh),0BjC1NfoEOOusryehmNudP,1
2,Movie,Joseph Williams,Don't Let Me Be Lonely Tonight,0CoSDzoNIKCRs124s9uTVy,3


### Filtros com máscaras booleanas e `query`

In [41]:
# Máscara booleana
filtro = (df['popularity'] >= 50) & (df['instrumentalness'] < 0.5)
df[filtro][['genre','popularity','energy','speechiness','instrumentalness']]

,genre,popularity,energy,speechiness,instrumentalness
135,R&B,65,0.689,0.1350,0.000000
136,R&B,63,0.610,0.0439,0.000000
137,R&B,62,0.520,0.0959,0.000004
138,R&B,61,0.366,0.1210,0.002430
139,R&B,68,0.621,0.0409,0.000000
...,...,...,...,...,...
232446,Soul,52,0.595,0.0431,0.109000
232480,Soul,51,0.753,0.0325,0.000004
232502,Soul,50,0.781,0.1250,0.000003
232513,Soul,52,0.422,0.0334,0.000000


In [42]:
df.columns

Index(['genre', 'artist_name', 'track_name', 'track_id', 'popularity',
       'acousticness', 'danceability', 'duration_ms', 'energy',
       'instrumentalness', 'key', 'liveness', 'loudness', 'mode',
       'speechiness', 'tempo', 'time_signature', 'valence'],
      dtype='object')

In [43]:
# query (requer nomes de colunas "seguros", sem espaços)
# Vamos criar uma cópia com colunas "seguras"
artistas = df.rename(columns={'artist_name': 'artist_name_'}).copy()
# Para usar query com números, primeiro convertemos a renda para float padrão.
artistas['artist_name_'] = artistas['artist_name_'].str.replace(' ', '_')
artistas.query("popularity >= 30 and energy > 0.5")[['artist_name_','genre','popularity','energy','speechiness']]

,artist_name_,genre,popularity,energy,speechiness
135,Mary_J._Blige,R&B,65,0.689,0.1350
136,Rihanna,R&B,63,0.610,0.0439
137,Yung_Bleu,R&B,62,0.520,0.0959
139,Olivia_O'Brien,R&B,68,0.621,0.0409
141,Nao,R&B,64,0.649,0.0875
...,...,...,...,...,...
232717,Belly,Soul,43,0.516,0.2130
232718,Muddy_Waters,Soul,43,0.739,0.0434
232720,Slave,Soul,39,0.714,0.0316
232721,Jr_Thomas_&_The_Volcanos,Soul,38,0.683,0.0337


### Ordenação e amostras

In [44]:
df.sort_values(by=['popularity', 'artist_name'], ascending=[False, True]).head()

,genre,artist_name,track_name,track_id,popularity,acousticness,danceability,duration_ms,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,time_signature,valence
9027,Dance,Ariana Grande,7 rings,14msK75pk3pA33pzPVNtBF,100,0.5780,0.725,178640,0.321,0.000000,C#,0.0884,-10.744,Minor,0.3230,70.142,4/4,0.319
107804,Pop,Ariana Grande,7 rings,14msK75pk3pA33pzPVNtBF,100,0.5780,0.725,178640,0.321,0.000000,C#,0.0884,-10.744,Minor,0.3230,70.142,4/4,0.319
9026,Dance,Ariana Grande,"break up with your girlfriend, i'm bored",4kV4N9D1iKVxx1KLvtTpjS,99,0.0421,0.726,190440,0.554,0.000000,F,0.1060,-5.290,Minor,0.0917,169.999,4/4,0.335
107802,Pop,Ariana Grande,"break up with your girlfriend, i'm bored",4kV4N9D1iKVxx1KLvtTpjS,99,0.0421,0.726,190440,0.554,0.000000,F,0.1060,-5.290,Minor,0.0917,169.999,4/4,0.335
86951,Rap,Post Malone,Wow.,6MWtB6iiXyIwun0YzU6DFP,99,0.1630,0.833,149520,0.539,0.000002,B,0.1010,-7.399,Minor,0.1780,99.947,4/4,0.385


## Selecionar Amostras aleatórias do dataset

In [45]:
df.sample(3, random_state=42)

,genre,artist_name,track_name,track_id,popularity,acousticness,danceability,duration_ms,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,time_signature,valence
788,Country,A Thousand Horses,My Time's Comin',16zol4GvHyTER5irYODUk0,45,0.00192,0.327,194107,0.8350,0.00015,C,0.1670,-4.952,Major,0.0609,171.795,4/4,0.3850
207109,Soundtrack,Mark Mothersbaugh,House Tour,6ac5gUfGTckpdGQCyWsdh2,25,0.93200,0.253,102920,0.0798,0.56800,C,0.0906,-18.512,Major,0.0439,110.931,4/4,0.0487
138644,Reggae,Unified Highway,We Can't Fall (Remix) [feat. J. Patz],09Yz6koF1Y15n1012t1UX6,19,0.03310,0.821,225437,0.7370,0.01340,E,0.1320,-6.295,Minor,0.2120,137.968,4/4,0.7870


### Agrupamento de Dados com `groupby` no pandas

#### O que é `groupby`?
- É um método do pandas usado para **agrupar linhas** com base em uma ou mais colunas.
- Permite aplicar funções de agregação (`sum`, `mean`, `count`, etc.) para cada grupo.

##### Analogia
Pense em `groupby` como:
1. **Dividir** os dados em grupos com base em um critério.
2. **Aplicar** uma função a cada grupo.
3. **Combinar** os resultados em um novo DataFrame ou Series.

### Sintaxe Básica
```python
df.groupby('coluna').funcao_agregacao()
df.groupby(['coluna1', 'coluna2']).funcao_agregacao()
```

### Média de popularidade por gênero

In [46]:
df.groupby("genre")["popularity"].mean().sort_values(ascending=False).head(10)

,popularity
genre,
Pop,66.590667
Rap,60.533795
Rock,59.619392
Hip-Hop,58.423131
Dance,57.275256
Indie,54.701561
Children’s Music,54.659040
R&B,52.308719
Alternative,50.213430


### Desvio padrão de Artista por Dançabilidade

In [47]:
df.groupby("artist_name")["danceability"].std().sort_values(ascending=False).head(10)

,danceability
artist_name,
POP ETC,0.331633
Erica Campbell,0.329512
Kin$oul,0.325976
This Will Destroy You,0.322441
Larry Grenadier,0.311127
Jaymay,0.310420
Mima,0.304056
Talvin Singh,0.301935
Giovanni Battista Pergolesi,0.301935


### Média de Gênero por Loudness e Speechiness

In [48]:
df.groupby("genre")[["loudness", "speechiness"]].mean().sort_values("genre", ascending=False).head(10)

,loudness,speechiness
genre,,
World,-10.705435,0.045766
Soundtrack,-19.282684,0.043852
Soul,-8.866409,0.082531
Ska,-6.172705,0.089158
Rock,-7.285875,0.053664
Reggaeton,-5.875960,0.127616
Reggae,-7.518107,0.116163
Rap,-6.669916,0.188186
R&B,-7.597064,0.120994


### Popularidade média por gênero e modo (Major/Minor)

In [49]:
df.groupby(["genre", "mode"])["popularity"].mean().reset_index()

,genre,mode,popularity
0,A Capella,Major,9.873563
1,A Capella,Minor,7.750000
2,Alternative,Major,49.993874
3,Alternative,Minor,50.594507
4,Anime,Major,24.465058
5,Anime,Minor,23.920213
6,Blues,Major,34.725331
7,Blues,Minor,34.786043
8,Children's Music,Major,4.304800
9,Children's Music,Minor,3.899281


## Exercícios — Introdução ao Pandas
1. Leitura e visualização inicial

    - Leia o arquivo melb_data.csv usando pandas.read_csv().

    - Mostre as 5 primeiras linhas do DataFrame.

2. Estatísticas descritivas

    - Mostre as estatísticas descritivas (describe()) apenas para as colunas Price e BuildingArea.

3. Filtro por bairro

    - Filtre apenas os imóveis localizados no CouncilArea "Melbourne".

    - Mostre as colunas Adress, Type e Price.

4. Ordenação

    - Ordene o DataFrame pelo preço (Price) em ordem decrescente.

    - Mostre apenas as 10 primeiras linhas.

5. Agrupamento

    - Agrupe os dados por ano de construção ('YearBuilt') e calcule o preço médio para cada ano.

## Referências e Leituras Recomendadas

- Documentação oficial do pandas (User Guide)  
- Leitura de dados: `read_csv`, `read_excel`, e guia de IO tools  
- Indexação/seleção: `.loc`, `.iloc`, máscaras, `query`  
- Funções básicas/essenciais